In [1]:
import pandas as pd
import boto3
from io import BytesIO

In [2]:
BUCKET_NAME = "itam-analytics-paulo"
SILVER_PREFIX = "vigila-canasta/silver/inegi/precios_promedio/cdmx/"

In [3]:
s3 = boto3.client("s3")

response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=SILVER_PREFIX
)

parquet_files = [
    obj["Key"]
    for obj in response.get("Contents", [])
    if obj["Key"].endswith(".parquet")
]

len(parquet_files), parquet_files[:5]

(21,
 ['vigila-canasta/silver/inegi/precios_promedio/cdmx/year=2024/month=08/INP_PP_CAB18a_2024_ago.parquet',
  'vigila-canasta/silver/inegi/precios_promedio/cdmx/year=2024/month=09/INP_PP_CAB18a_2024_sep.parquet',
  'vigila-canasta/silver/inegi/precios_promedio/cdmx/year=2024/month=10/INP_PP_CAB18a_2024_oct.parquet',
  'vigila-canasta/silver/inegi/precios_promedio/cdmx/year=2024/month=11/INP_PP_CAB18a_2024_nov.parquet',
  'vigila-canasta/silver/inegi/precios_promedio/cdmx/year=2024/month=12/INP_PP_CAB18a_2024_dic.parquet'])

In [4]:
dfs = []

for key in parquet_files:
    obj = s3.get_object(Bucket=BUCKET_NAME, Key=key)
    df_temp = pd.read_parquet(BytesIO(obj["Body"].read()))
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)

df.head()

,fecha,anio,mes,subclase,generico,especificacion,producto_id,precio_promedio,cantidad,unidad,source_file,source_s3_key,load_timestamp
0,2024-08-01,2024,8,04 Arroz y cereales preparados,Arroz,"VERDE VALLE, BLANCO, SUPER EXTRA, BOLSA DE 900 G",26fb5c81c836b16935a4048b04052f98,44.44,1,KG,INP_PP_CAB18a_2024_ago.CSV,vigila-canasta/bronze/inegi/precios_promedio/c...,2026-05-21T15:08:39.974453+00:00
1,2024-08-01,2024,8,04 Arroz y cereales preparados,Arroz,"VERDE VALLE, BLANCO, SUPER EXTRA, BOLSA DE 1 KG",da271c75a8b27a5b75ae2e85c5ad032b,38.75,1,KG,INP_PP_CAB18a_2024_ago.CSV,vigila-canasta/bronze/inegi/precios_promedio/c...,2026-05-21T15:08:39.974453+00:00
2,2024-08-01,2024,8,04 Arroz y cereales preparados,Arroz,"SOS, BLANCO, EXTRA, BOLSA DE 1 KG",81e7bd7aead7648e5330dba54a3b2763,32.00,1,KG,INP_PP_CAB18a_2024_ago.CSV,vigila-canasta/bronze/inegi/precios_promedio/c...,2026-05-21T15:08:39.974453+00:00
3,2024-08-01,2024,8,04 Arroz y cereales preparados,Arroz,"MP, BLANCO, SUPER EXTRA, BOLSA DE 900 G",b5f5d5f90492e1555f3bb5858520dfec,22.22,1,KG,INP_PP_CAB18a_2024_ago.CSV,vigila-canasta/bronze/inegi/precios_promedio/c...,2026-05-21T15:08:39.974453+00:00
4,2024-08-01,2024,8,04 Arroz y cereales preparados,Arroz,"VERDE VALLE, BLANCO, SUPER EXTRA, BOLSA DE 900 G",26fb5c81c836b16935a4048b04052f98,43.17,1,KG,INP_PP_CAB18a_2024_ago.CSV,vigila-canasta/bronze/inegi/precios_promedio/c...,2026-05-21T15:08:39.974453+00:00


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 159680 entries, 0 to 159679
Data columns (total 13 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   fecha            159680 non-null  datetime64[us]
 1   anio             159680 non-null  Int64         
 2   mes              159680 non-null  Int64         
 3   subclase         159680 non-null  str           
 4   generico         159680 non-null  str           
 5   especificacion   159680 non-null  str           
 6   producto_id      159680 non-null  str           
 7   precio_promedio  159680 non-null  float64       
 8   cantidad         159680 non-null  int64         
 9   unidad           159680 non-null  str           
 10  source_file      159680 non-null  str           
 11  source_s3_key    159680 non-null  str           
 12  load_timestamp   159680 non-null  str           
dtypes: Int64(2), datetime64[us](1), float64(1), int64(1), str(8)
memory usage: 57.7 MB


In [6]:
df.shape

(159680, 13)

In [7]:
df["fecha"].min(), df["fecha"].max()

(Timestamp('2024-08-01 00:00:00'), Timestamp('2026-04-01 00:00:00'))

In [8]:
df[["subclase", "generico", "especificacion", "producto_id"]].nunique()

subclase            76
generico           292
especificacion    8721
producto_id       8891
dtype: int64

In [9]:
df.isna().sum().sort_values(ascending=False)

fecha              0
anio               0
mes                0
subclase           0
generico           0
especificacion     0
producto_id        0
precio_promedio    0
cantidad           0
unidad             0
source_file        0
source_s3_key      0
load_timestamp     0
dtype: int64

In [10]:
df["subclase"].value_counts().head(20)

subclase
17 Hortalizas frescas                                11592
16 Frutas frescas                                    10899
55 Medicamentos                                       5292
13 Derivados de leche                                 5061
73 Otros artículos de esparcimiento                   4893
36 Ropa de abrigo                                     4452
07 Carne y vísceras de res                            4179
30 Pantalones, trajes y otras prendas para hombre     4158
34 Ropa para niños                                    4074
49 Aparatos eléctricos                                3969
71 Otros servicios de esparcimiento                   3906
33 Vestidos, faldas y conjuntos para mujer            3591
53 Accesorios textiles de uso en el hogar             3528
67 Educación privada                                  3360
74 Restaurantes, bares y similares                    3087
22 Refrescos envasados y agua embotellada             2982
51 Accesorios domésticos                       

In [11]:
df["generico"].value_counts().head(20)

generico
Ropa de abrigo                                          4452
Carne de res                                            3927
Otras prendas de vestir para hombre                     3255
Otras prendas de vestir para mujer                      2730
Otras frutas                                            2646
Ropa interior para niños, niñas y adolescentes          2541
Pescado                                                 2394
Toallas, cortinas y otros blancos                       2373
Carne de cerdo                                          2184
Restaurantes y similares                                2163
Pollo                                                   2037
Otros medicamentos                                      1848
Instrumentos musicales, y descargas de audio y video    1806
Yogurt                                                  1680
Limón                                                   1617
Calabacita                                              1596
Bolsas y mochil